In [ ]:
# Install if needed:
# !pip install pandas nltk scikit-learn matplotlib

import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

nltk.download('stopwords')

# 1. Load dataset
df = pd.read_csv("20000i.csv")
print(df.shape)
print(df.head())

# 2. NLP cleaning
stop = set(stopwords.words("english"))

def clean(x):
    x = re.sub(r'[^a-zA-Z\s]', '', str(x).lower())
    return ' '.join(w for w in x.split() if w not in stop)

df["clean_text"] = df["text"].fillna("").apply(clean)

# 3. TF-IDF
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df["clean_text"])

# 4. K-Means
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = model.fit_predict(X)

# 5. Show clusters
for i in range(k):
    print("\nCLUSTER", i)
    print(df[df["cluster"] == i]["text"].head(5).to_string(index=False))

# 6. Visualisation
svd = TruncatedSVD(n_components=2, random_state=42)
X2 = svd.fit_transform(X)

plt.scatter(X2[:,0], X2[:,1], c=df["cluster"], alpha=0.5)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("NLP K-Means Clustering")
plt.show()

# 7. Save results
df.to_csv("20000i_kmeans_results.csv", index=False)